# 3b — Google Maps Traffic

Renders a Google Maps traffic screenshot with Playwright, classifies pixels
by CIE76 color distance, and bearing-based KD-tree matching to OSM edges.
Writes congestion to `google_congestion_history`.

Requires `GOOGLE_MAPS_API_KEY` (Maps JavaScript API) in `../.env`.

---

## How it works

Google Maps does not expose a tile API for traffic data. Instead we render the page
with a headless Chromium browser (Playwright), capture a screenshot, and interpret
the pixel colors.

**Step 1 — Render:** launch a custom HTML page that shows only roads (white) and the
TrafficLayer (colored). The page is sized to exactly cover the boundary bounding box
at the target zoom level.

**Step 2 — Classify pixels:** each non-white pixel is classified by its CIE76 distance
to four reference colors from the Google Maps JS API:

| Level | Hex | RGB |
|---|---|---|
| `low` | `#11D68F` | 17, 214, 143 |
| `moderate` | `#FFCF43` | 255, 207, 67 |
| `heavy` | `#F24E42` | 242, 78, 66 |
| `severe` | `#A92727` | 169, 39, 39 |

Pixels farther than 25 CIE76 units from all references are discarded (background/UI noise).

**Step 3 — Bearing-based match:** traffic pixels form colored stripes along roads.
For each pixel, we estimate the stripe direction via PCA on nearby traffic pixels,
then match to the OSM edge whose bearing aligns within **40°**.
Ambiguous matches (two different OSM ways both align) are discarded.

**Key difference from Mapbox:** Google only colors *congested* roads — `no data` in
Google usually means normal flow, not missing data.

In [ ]:
%pip install Pillow playwright scipy python-dotenv folium --quiet
!playwright install chromium --quiet

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, '..')
from scripts.google_traffic import GoogleTraffic
from scripts.traffic_db import TrafficDB, CONGESTION_COLORS

load_dotenv(Path('../.env'), override=True)

# ── Configuration ─────────────────────────────────────────────────────
NAME     = 'sodermalm'
ZOOM     = 16        # ~2.4 m/px; auto-reduced if screenshot exceeds 160 MP
DB       = f'../db/{NAME}.duckdb'
BOUNDARY = f'../boundaries/{NAME}.geojson'
API_KEY  = os.environ.get('GOOGLE_MAPS_API_KEY', '')
# ─────────────────────────────────────────────────────────────────────

if not API_KEY:
    raise ValueError('GOOGLE_MAPS_API_KEY not set in ../.env')
print(f'API key  : {API_KEY[:8]}...{API_KEY[-4:]}')
print(f'DuckDB   : {DB}')
print(f'Boundary : {BOUNDARY}')
print(f'Zoom     : {ZOOM}  (~{40_075_016 / (256 * 2**ZOOM):.2f} m/px)')

---
## Step 1 — Render screenshot + classify + map match

All three sub-steps run inside `traffic.map_match()`:

1. Playwright renders the Google Maps page (~5–30 s depending on boundary size)
2. Every pixel is classified by CIE76 distance to the four reference colors
3. KD-tree bearing match assigns pixels to OSM edges

**Matching parameters:**
- `MATCH_BUFFER_M = 10` — edge sample point must be within 10 m of the pixel
- `NEIGHBOR_RADIUS_M = 25` — use 25 m neighborhood to estimate stripe direction via PCA
- `BEARING_THRESHOLD = 40°` — stripe direction vs edge bearing must differ by ≤ 40°
- `MIN_NEIGHBORS = 5` — need at least 5 nearby pixels to reliably estimate stripe direction

Expected coverage: **5–15%** of edges for Google (only congested roads are colored).

In [ ]:
%%time
traffic = GoogleTraffic(api_key=API_KEY, zoom=ZOOM)
edge_cong, total_pixels = traffic.map_match(DB, BOUNDARY)

from collections import Counter
print(f'Traffic pixels  : {total_pixels:,}')
print(f'Edges matched   : {len(edge_cong):,}')
print(f'Distribution    : {Counter(edge_cong.values())}')

---
## Step 2 — Write to DuckDB

`n_segments` stores the total traffic pixel count — useful for comparing snapshot density
across runs (more pixels = more congestion at fetch time).

In [ ]:
%%time
with TrafficDB(DB, read_only=False) as db:
    run_id = db.write_congestion(
        edge_cong,
        source        = 'google',
        zoom          = ZOOM,
        n_segments    = total_pixels,
        boundary_name = NAME,
    )
print(f'Written as run_id={run_id}  source=google  zoom={ZOOM}')

---
## Step 3 — Inspect results

In [ ]:
%%time
with TrafficDB(DB) as db:
    print('=== History ===')
    display(db.get_history_index())
    print('\n=== Congestion summary (Google) ===')
    display(db.get_congestion_summary('google'))

    edges = db.get_edges(source='google')
    m = db.plot_edges(edges)
m

---
## Step 4 — Compare Mapbox vs Google (optional)

Mapbox colors all roads (`low` = normal flow); Google colors only congested roads.
So `Mapbox=low, Google=no data` is the expected result for free-flowing roads — both agree,
they just express it differently.

Requires Mapbox data to also be present (run notebook 3a first).

In [ ]:
with TrafficDB(DB) as db:
    db.plot_comparison(sources=['mapbox', 'google'])